# 第 4 讲：探索性数据分析与可视化表达

> 从图表中发现模式，并训练把图表转成建模假设和文字解释。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-04"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

In [ ]:
n = 180
df = pd.DataFrame({
    "income": rng.normal(6000, 1200, n),
    "education_years": rng.normal(15, 2.2, n),
    "commute_time": rng.normal(42, 15, n),
    "satisfaction": rng.normal(70, 8, n),
})
df["satisfaction"] += 0.002 * df["income"] + 1.2 * (df["education_years"] - 15) - 0.18 * df["commute_time"]
df["commute_time"] = df["commute_time"].clip(lower=5)
df["satisfaction"] = df["satisfaction"].clip(lower=0, upper=100)
corr = df.corr(numeric_only=True)
corr.to_csv(OUTPUT_ROOT / "eda_correlation.csv", encoding="utf-8-sig")
corr

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
axes[0].set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
axes[0].set_yticks(range(len(corr.columns)), corr.columns)
axes[0].set_title("Correlation heatmap")
fig.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].scatter(df["commute_time"], df["satisfaction"], alpha=0.7)
axes[1].set_title("Commute time vs satisfaction")
axes[1].set_xlabel("Commute time")
axes[1].set_ylabel("Satisfaction")
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "eda_story.png", dpi=160)
plt.show()

In [ ]:
interpretation = pd.DataFrame({
    "图表": ["Correlation heatmap", "Commute scatter"],
    "观察": ["satisfaction 与 commute_time 呈负相关", "通勤时间越长，满意度整体越低"],
    "建模启发": ["后续模型应纳入 commute_time", "可以检验通勤时间的边际影响"],
})
interpretation.to_csv(OUTPUT_ROOT / "figure_interpretation.csv", index=False, encoding="utf-8-sig")
interpretation